# Notebook 4 — Reasoning About the Semantics

**Covers Chapter 2 §2.5–2.8** plus the quiz, plus the bridge from your Python intuition to exam-style formal answers.

Topics:

- The reflexive-transitive closure `⇒*` and what it means to terminate.
- **Proposition 2** — structural decomposition of `S; T` traces.
- **Proposition 3** — associativity of sequential composition.
- **Exercise 17** — state preservation for variables not mentioned in S.
- **Exercise 18** — determinism (the next state is unique given the next program).
- The 3-question quiz.
- **The translation section** — how to take an interpreter run and convert it into the kind of `⇒ⁿ` derivation the exam expects.

## What you'll be able to do by the end

- Argue (informally but rigorously) about properties of the small-step semantics by case-analysis on rules.
- Predict whether a given While program terminates, and explain why.
- Convert any interpreter trace into the chapter's compact `⇒ⁿ` exam style.
- Sit the exam without freezing on a trace question.

## On the proof-style exercises

Exercises 17 and 18 are different from earlier ones — they ask you to *reason about* the semantics, not just apply it. My solutions match the chapter's style as closely as possible, but proof exercises sometimes have multiple valid presentations. **Verify with a GTA** if you want to be sure your phrasing would be marked correct.

In [ ]:
from while_lang import (
    parse, run, trace,
    A, B, step, step_iter,
    Config, Transition, StepBudgetExceeded,
    check_state, check_steps,
)

## §1. The reflexive-transitive closure `⇒*`

We've been writing `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` for **one step**. To talk about **many steps** the chapter introduces:

- `⟨S, σ⟩ ⇒ⁿ ⟨S', σ'⟩` — there exist intermediate configurations such that we go from one to the other in *exactly n* steps.
- `⟨S, σ⟩ ⇒* ⟨S', σ'⟩` — *some* number of steps (possibly zero), aka the **reflexive-transitive closure**.

The reflexive part: `⟨S, σ⟩ ⇒* ⟨S, σ⟩` for any S, σ — zero steps gets you nowhere, but that's still in the closure.

**Key fact** (chapter §2.3.3, end): given a While program S and a state σ, the sequence of computation steps is unique. Either it goes forever, or there's a unique state σ' with `⟨S, σ⟩ ⇒* ⟨skip, σ'⟩`.

This is what makes While *deterministic*: the next configuration is always uniquely determined.

## §2. Proposition 2 — structural decomposition of `S; T` traces

**Statement (paraphrased):** if `⟨S; T, σ⟩ ⇒ⁿ ⟨U, τ⟩`, then exactly one of these holds:

1. **Still inside S:** there's a U' such that U = U'; T, and `⟨S, σ⟩ ⇒ⁿ ⟨U', τ⟩`.
2. **Done with S, working on T:** there's a state σ' and natural numbers k, l with k + l + 1 = n such that `⟨S, σ⟩ ⇒ᵏ ⟨skip, σ'⟩` and `⟨T, σ'⟩ ⇒ˡ ⟨U, τ⟩`.

**What this says intuitively:** when you execute `S; T` step by step, *first* you execute all of S, *then* all of T. There's a sharp boundary — at any point, either you're still in S or you've moved on to T (with one extra step for the `skip-;` rule that drops the finished S).

**Why this matters:** it formalizes "do one thing, then another." The semantics doesn't permit S and T to interleave or T to start running before S finishes.

### The proof in one paragraph

Induction on n. Base case n = 0: U = S; T (zero steps), so case 1 holds with U' = S. Step case n + 1: by definition there's a U₀ with `⟨S; T, σ⟩ ⇒ ⟨U₀, ρ⟩ ⇒ⁿ ⟨U, τ⟩`. The single step from `⟨S; T, σ⟩` is determined by which rule applies:

- If S is `skip`, the `skip-;` rule fires: U₀ = T, ρ = σ. Then the remaining n steps go through T, and we're in case 2 with k = 0, l = n.
- Otherwise, the general `;` rule fires: U₀ = S₁; T for some S₁ with `⟨S, σ⟩ ⇒ ⟨S₁, ρ⟩`. Apply IH to `⟨S₁; T, ρ⟩ ⇒ⁿ ⟨U, τ⟩`. Either case 1 (still in S₁ part of S₁; T, hence still in S overall) or case 2 (S₁ finished and T took some steps).

The chapter has the full version. We reproduce its structure here because it's the kind of induction expected in exam-style proofs.

## §3. Proposition 3 — associativity of `;`

**Statement:** `⟨(S; T); U, σ⟩ ⇒* ⟨skip, τ⟩` if and only if `⟨S; (T; U), σ⟩ ⇒* ⟨skip, τ⟩`.

**What this says:** how you parenthesize a chain of `;` doesn't matter for termination or final state. So even though the BNF grammar is ambiguous about whether `S₁; S₂; S₃` means `(S₁; S₂); S₃` or `S₁; (S₂; S₃)`, the *behavior* is the same either way.

### Why associativity follows from Proposition 2

The `(⇒)` direction. Assume `⟨(S; T); U, σ⟩ ⇒* ⟨skip, τ⟩`. By Prop 2 (applied to the outer `;`): there's a state σ' with `⟨S; T, σ⟩ ⇒* ⟨skip, σ'⟩` and `⟨U, σ'⟩ ⇒* ⟨skip, τ⟩`.

Apply Prop 2 again, to the inner `;`: there's a state σ'' with `⟨S, σ⟩ ⇒* ⟨skip, σ''⟩` and `⟨T, σ''⟩ ⇒* ⟨skip, σ'⟩`.

Now combine: starting from σ, S terminates in σ''. Then T starts from σ'' and terminates in σ'. Then U starts from σ' and terminates in τ. By Prop 2 (other direction), this means `⟨T; U, σ''⟩ ⇒* ⟨skip, τ⟩`. Combined with `⟨S, σ⟩ ⇒* ⟨skip, σ''⟩`, by Prop 2 again: `⟨S; (T; U), σ⟩ ⇒* ⟨skip, τ⟩`. ✓

The other direction (`⇐`) is symmetric.

### Sanity check by example

In [ ]:
# Three statements; pick a state; both groupings should produce same final state.
left_grouped  = '(x := 1; y := 2); z := 3'
right_grouped = 'x := 1; (y := 2; z := 3)'

for state in [{}, {'a': 99}]:
    L = run(left_grouped, state)
    R = run(right_grouped, state)
    print(f'state={state}:  left={L},  right={R},  equal={L == R}')

## Exercise 17 — state preservation for unmentioned variables

> Reason that if S is a statement that does not mention the variable x then given a state σ we have that ⟨S, σ⟩ ⇒ ⟨S', σ'⟩ implies that σ(x) = σ'(x).

**The claim:** running one step of a program that doesn't mention `x` cannot change the value of `x`.

### Proof — case analysis on which rule fires

Since `S ≠ skip` (otherwise no transition), exactly one of the rules applies. Examine each.

**Case 1 (assignment):** `S = y := a` for some variable y and arithmetic expression a. The transition is `⟨y := a, σ⟩ ⇒ ⟨skip, σ[y ↦ A a σ]⟩`. Since S doesn't mention `x`, neither y nor any variable in a is `x`. So y ≠ x. Hence:

$$\sigma'(x) = (\sigma[y \mapsto \mathcal{A}\,a\,\sigma])(x) = \sigma(x) \quad \text{(by definition of update, since } y \neq x\text{)}$$

**Case 2 (skip-;):** `S = skip; T`. The transition is `⟨skip; T, σ⟩ ⇒ ⟨T, σ⟩`. State unchanged, so σ'(x) = σ(x). ✓

**Case 3 (general ;):** `S = S₁; S₂` where S₁ ≠ skip. The transition fires only if `⟨S₁, σ⟩ ⇒ ⟨S₁', σ'⟩`. Since S doesn't mention x, neither does S₁. By induction on the size of the program (S₁ is strictly smaller than S), σ'(x) = σ(x). ✓

**Case 4 (if-tt or if-ff):** `S = if b then S₁ else S₂`. Either rule leaves σ unchanged (state stays σ, only the program changes to S₁ or S₂). So σ'(x) = σ(x). ✓

**Case 5 (while-tt or while-ff):** `S = while b do S₁`. Either rule leaves σ unchanged. So σ'(x) = σ(x). ✓

**Conclusion:** in every case, σ(x) = σ'(x). ∎

### The contrast — what happens to mentioned variables?

If S *does* mention x, the value of x is unconstrained. A single transition can:

- Leave it alone (e.g. if-rule firing — state preserved).
- Change it by an assignment.
- Change it as part of a deeper transition through a `;` rule.

There's no general bound — a single assignment can set x to anything in ℤ. The chapter doesn't ask us to characterize this further; the point of the exercise is the contrapositive.

In [ ]:
# Empirical sanity: take a program that doesn't mention x, run one step,
# and check that x's value is unchanged.
from while_lang import Config
cfg = Config(parse('y := 5; z := y + 1'), {'x': 99, 'y': 0})   # S doesn't mention x
for step_no in range(1, 8):
    t = step(cfg)
    if t is None:
        print(f'step {step_no}: terminated, x = {cfg.state.get("x", 0)}')
        break
    print(f'step {step_no}: x = {t.after.state.get("x", 0)} (rule {t.rule})')
    cfg = t.after

## Exercise 18 — the next state is unique

> Justify the claim: if `⟨S, σ⟩ ⇒ ⟨S', ρ⟩` and `⟨S, σ⟩ ⇒ ⟨S', τ⟩`, then ρ = τ.

**What this says:** if you take one step of S in σ, and the program left to execute (S') is the same in both cases, then the resulting state is the same. In other words: given the start (S, σ) and the program-after (S'), the state-after is uniquely determined.

### Proof — case analysis on which rule fires

Each rule is **deterministic** (given the rule, the result is fully determined by σ). And the *choice of rule* is determined by S. So the result is uniquely determined.

**Case 1 (assignment):** S = y := a. The only rule is `:=`. Result: `⟨skip, σ[y ↦ A a σ]⟩`. Since `A` is a function (deterministic), σ[y ↦ A a σ] is uniquely determined. So ρ = τ. ✓

**Case 2 (skip-;):** S = skip; T. The only rule is `skip-;`. Result: `⟨T, σ⟩`. State is just σ — unique. So ρ = σ = τ. ✓

**Case 3 (general ;):** S = S₁; S₂ with S₁ ≠ skip. The only rule applicable is the general `;` rule, which delegates to whatever rule fires for `⟨S₁, σ⟩`. By induction on the size of S, that subordinate rule produces a unique state. So ρ = τ. ✓

**Case 4 (if):** S = if b then S₁ else S₂. The rule that fires depends on `B(b, σ)` — but **both branches leave σ unchanged**. So whether `if-tt` or `if-ff` fires, the state in the result is σ. Hence ρ = σ = τ. ✓

(Note: S' is *also* uniquely determined here — it's S₁ if the condition holds, S₂ otherwise. But the exercise only asks about ρ = τ given S' is fixed; the case analysis still works.)

**Case 5 (while):** S = while b do S₁. Same as case 4 — both rule variants leave σ unchanged. ρ = σ = τ. ✓

**Conclusion:** in every case, ρ = τ. ∎

### A stronger claim, for free

The proof above incidentally establishes more: not only is ρ determined given S', but **S' is itself uniquely determined given S, σ**. So the small-step relation is a *partial function* — from (S, σ) you get at most one (S', σ'). This is what makes While deterministic.

(Some semantics — for non-deterministic or concurrent languages — explicitly relax this. While doesn't.)

## §4. The quiz — three questions

Walking through the questions in `quiz.txt`.

### Q1 — `while 1=2 do (x:=tt)`: not syntactically valid?

> **Assertion:** the program is not syntactically valid. **Reason:** `1=2` will always be false.

**Is the assertion true?** Yes — but for a different reason than the one given. The bug is `x := tt`. The grammar says assignment is `<var> := <aexp>`, and `tt` is a *boolean* expression, not arithmetic. So you can't assign `tt` to a variable.

**Is the reason true?** Yes — `1 = 2` is indeed always false.

**Does the reason justify the assertion?** **No.** Always-false conditions don't make a program syntactically invalid — there's nothing wrong with `while ff do (x := 1)`, even though it's a no-op.

**Answer:** the assertion and reason are **both correct but the reason is invalid** — it doesn't actually justify the assertion.

### Q2 — `while true do (x := x + 1)`: valid?

Strictly speaking the keyword should be `tt`, not `true` — the BNF doesn't include `true`. But the quiz's intended reading takes `true` as casual shorthand for `tt`. Then the question becomes: is a program that loops forever "valid"?

**Yes — it's valid.** The syntax says nothing about whether the loop terminates. "Loops forever" is a *semantic* property, not a *syntactic* one. The chapter's whole point in §2.5 is that non-terminating programs are perfectly well-formed; we just have to reason about whether they terminate as a separate matter.

**Answer:** **is a valid program.** It's the programmer's job to think about whether it'll terminate; the syntax can't help with that.

### Q3 — trace this program

```
x := 1;
if x = 0 then
    y := 1
else
    y := 2
while y <= 2 do
    x := x + 1;
    y := y + 1
```

Per the feedback hint: "the `else` branch is selected and then the body of the loop is executed once." So:

1. `x := 1` → x = 1.
2. `if x = 0`: condition is false (x is 1), so else-branch fires: `y := 2`.
3. `while y <= 2`: 2 ≤ 2 is true, body runs: `x := x + 1` makes x = 2; `y := y + 1` makes y = 3.
4. Check `y <= 2` again: 3 ≤ 2 is false, loop exits.

**Final values: x = 2, y = 3.**

In [ ]:
# Verify Q3 by running the program. We need to add brackets around the if-else and the while body
# (the original program text is missing them, which would actually be a parse error per the formal
# grammar — but the quiz's intent is clear from the feedback).
quiz_q3 = '''
    x := 1;
    if x = 0 then
        y := 1
    else (
        y := 2
    );
    while y <= 2 do (
        x := x + 1;
        y := y + 1
    )
'''

print(run(quiz_q3, {}))

## §5. The bridge — Python intuition → exam-style formal answers

The whole point of building this interpreter was so you'd see formal traces produced by code you understand. But on the exam, you have to *write* those traces by hand — and the chapter uses a more compact convention than what our interpreter prints.

This section walks three programs through both forms side by side, so you can see the mapping.

### Translation rules

1. **Combine adjacent `:=`, `;`, and `skip-;` rules** into one super-step labelled `⇒ⁿ` for some small n. The chapter does this all the time (see Example 7's `⇒⁴` annotations).
2. **Drop the bracketed rule names** when the rule is obvious from context. Show them when ambiguity could arise (e.g. `while-tt` vs `while-ff`, `if-tt` vs `if-ff`).
3. **Use the `L` abbreviation** for the body of a `while` loop after first introduction. This is what the chapter does and what our `view='formal'` does.
4. **State descriptions in curly braces** with `↦` arrows; omit zero-valued variables you didn't explicitly assign.

### Bridge example 1 — tiny program

Program: `x := 1; y := 2`. Start state: `σ = {}`.

In [ ]:
tiny = 'x := 1; y := 2'
print(trace(tiny, {}, view='formal'))

**Interpreter output (4 transitions): every micro-step is shown.**

**Exam-style answer (collapsed):**

```
⟨x := 1; y := 2, σ⟩
  ⇒²  ⟨y := 2, σ[x ↦ 1]⟩          (assignment + skip-;)
  ⇒²  ⟨skip, σ[x ↦ 1, y ↦ 2]⟩      (assignment + (vacuous) skip-;)
```

Two super-steps instead of four, with rule names abbreviated. **This is the level of compression the chapter uses in Example 7.**

### Bridge example 2 — small loop

Program: `x := 0; while x <= 2 do (x := x + 1)`. Start: `σ = {}`.

In [ ]:
loop = '''
    x := 0;
    while x <= 2 do (
        x := x + 1
    )
'''
print(trace(loop, {}, view='formal'))

**Exam-style version**, using `L` for `while x ≤ 2 do (x := x + 1)`:

```
⟨x := 0; L, σ⟩
  ⇒²  ⟨L, σ[x ↦ 0]⟩                                    (:= and skip-;)
  ⇒  ⟨x := x + 1; L, σ[x ↦ 0]⟩                          (while-tt, since 0 ≤ 2)
  ⇒²  ⟨L, σ[x ↦ 1]⟩                                    (:= and skip-;)
  ⇒  ⟨x := x + 1; L, σ[x ↦ 1]⟩                          (while-tt, since 1 ≤ 2)
  ⇒²  ⟨L, σ[x ↦ 2]⟩                                    (:= and skip-;)
  ⇒  ⟨x := x + 1; L, σ[x ↦ 2]⟩                          (while-tt, since 2 ≤ 2)
  ⇒²  ⟨L, σ[x ↦ 3]⟩                                    (:= and skip-;)
  ⇒  ⟨skip, σ[x ↦ 3]⟩                                   (while-ff, since ¬(3 ≤ 2))
```

Note how the `while-tt` and `while-ff` rules **stay visible** because the choice between them is the interesting bit. The administrative `:=`, `;`, `skip-;` get bundled into `⇒²` super-steps.

### Bridge example 3 — nested if-in-while

Program: `x := 0; while x <= 3 do (if x = 2 then (y := 99) else (skip); x := x + 1)`. Start: `σ = {}`.

In [ ]:
nested = '''
    x := 0;
    while x <= 3 do (
        if x = 2 then
            y := 99
        else (
            skip
        );
        x := x + 1
    )
'''

# This is long. Use the table view for readability.
print(trace(nested, {}, view='table'))

**Exam-style table** — the same execution but only showing one row per loop iteration ("loop" entries) plus the if-firings:

```
        x   y     condition x ≤ 3
start   0   0
loop    0   0     tt          (enter body, if-ff fires since x ≠ 2)
loop    1   0     tt
loop    2   0     tt          (if-tt fires: y := 99)
loop    2   99    
loop    3   99    tt
loop    4   99    ff          (exit)
```

This is the format the chapter wants in Examples 8 and 11. Compare it to our interpreter's full table — same information, much more compact.

### Translation summary

| What the interpreter shows | What the exam wants |
|---|---|
| Every transition with rule name | Multi-step `⇒ⁿ` collapses for routine rules |
| Full state on every line | State *deltas* via `σ[x ↦ v]` notation |
| Verbose table with every step | Compact table, one row per iteration / branch decision |
| `[skip-;]`, `[;]` shown | Implicit; mention rule only when interesting |
| L-abbreviation for repeated whiles | Same — chapter does this |

**Practice routine for the exam:** run a program in the interpreter, look at the formal trace, then re-write it on paper in the chapter's compressed style. Do this 3–5 times for representative programs (one short, one with a loop, one with nested control flow) and you'll have the translation as muscle memory.

## Summary — what you have now

**Across the four notebooks:**

- **N1** — the executable interpreter, including the predict-cell harness.
- **N2** — full syntactic analysis: BNF, all 6 chapter examples, exercises 1–10 (8/9 partially, depending on missing references).
- **N3** — operational semantics: states, A/B, the small-step rules, all 5 chapter examples 7–11, exercises 11–16 (the latter being the showpiece Diophantine derivation).
- **N4** — reasoning: Propositions 2 & 3, exercises 17–18, the quiz, and the Python-to-formal-exam-notation bridge.

**What you should be able to do now:**

- Read `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` natively without translating to Python.
- Produce a state-tracking table by hand for any small program.
- Identify which rule fires at any point in a derivation.
- Walk through Exercise 14's gcd derivation without referring to the chapter.
- Articulate the structure of a proof by case-analysis on rules (exercises 17, 18).
- Convert an interpreter run into the chapter's compact `⇒ⁿ` exam style.

**What I cannot promise:**

- Exercises 8 and 9 — the source material (Sections 6.2.1 / 6.2.2) wasn't in the provided files. **Verify with a GTA.**
- The exact phrasing a Manchester examiner would use for the proof-style answers (15, 17, 18). My versions match the chapter's structure but "correct phrasing" is taste-dependent. **Verify with a GTA.**
- That you've actually grokked it. That part is on you. The interpreter and the predict cells are the calibration tools — if a `check_state` keeps coming back red, that's the exact place to slow down and re-read.

Good luck.